# Bank Marketing Campaign — Term Deposit Subscription Prediction

**Goal:** Predict whether a bank client will subscribe to a term deposit (`y`) based on a Portuguese bank's direct marketing campaign data.

**Dataset:** 37,069 records, 20 features — demographic, financial, and campaign contact attributes. Source: [UCI Bank Marketing Dataset](https://archive.ics.uci.edu/ml/datasets/bank+marketing) via BYU-Idaho CSE 450.

**Approach:**
- EDA with `lets-plot` visualizations
- Feature encoding (ordinal education, OHE for categoricals, binary contact flag)
- Random Forest classifier with cross-validated hyperparameter tuning
- Threshold optimization for business-relevant recall/precision tradeoff

**Class imbalance note:** ~11% positive rate → accuracy is a misleading metric. Model is evaluated on F1-score and AUC-ROC.


## 0. Environment Setup

In [1]:
# Install lets-plot if running on Colab / fresh environment
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lets-plot', '--quiet'], check=True)


CompletedProcess(args=['c:\\Users\\alexa\\miniconda3\\python.exe', '-m', 'pip', 'install', 'lets-plot', '--quiet'], returncode=0)

## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lets_plot import (
    ggplot, aes, geom_boxplot, geom_bar, geom_histogram,
    ggtitle, xlab, ylab, theme_bw, LetsPlot
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, precision_recall_curve, ConfusionMatrixDisplay
)

LetsPlot.setup_html()

# ── Load data ─────────────────────────────────────────────────────────────────
campaign = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank.csv')
print(f"Shape: {campaign.shape}")
campaign.head()


## 2. Exploratory Data Analysis

In [ ]:
# ── Basic overview ────────────────────────────────────────────────────────────
print("Dataset info:")
campaign.info()
print("\nDescriptive stats:")
campaign.describe()


In [ ]:
# ── Target class balance ──────────────────────────────────────────────────────
target_counts = campaign['y'].value_counts()
print("Target distribution:")
print(target_counts)
print(f"\nPositive rate: {target_counts['yes'] / len(campaign):.2%}")
print("→ Class imbalance present — accuracy alone is misleading. Use F1 / AUC-ROC.")


In [ ]:
# ── Age distribution by subscription status ──────────────────────────────────
plot_age = (
    ggplot(campaign, aes(x='y', y='age', fill='y'))
    + geom_boxplot()
    + ggtitle('Age Distribution by Subscription Status')
    + xlab('Subscription Status') + ylab('Age')
    + theme_bw()
)
plot_age.show()


In [ ]:
# ── Campaign contact frequency ────────────────────────────────────────────────
# Note: pdays=999 means client was not previously contacted.
print(f"pdays=999 (never contacted): {(campaign['pdays'] == 999).mean():.2%}")
print("Mean contacts for subscribers vs. non-subscribers:")
print(campaign.groupby('y')['campaign'].mean())


In [ ]:
# ── Subscription rate by job type ─────────────────────────────────────────────
job_rate = (
    campaign.groupby('job')['y']
    .apply(lambda x: (x == 'yes').mean())
    .sort_values(ascending=False)
    .reset_index()
)
job_rate.columns = ['job', 'subscription_rate']
print(job_rate.to_string(index=False))


In [ ]:
# ── Economic indicators ───────────────────────────────────────────────────────
# euribor3m and emp.var.rate are highly correlated macro signals.
print("Correlation between economic indicators:")
econ_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
print(campaign[econ_cols].corr().round(2))


## 3. Feature Engineering & Preprocessing

| Feature | Treatment | Rationale |
|---|---|---|
| `education` | Ordinal encoding | Natural order: illiterate < basic.4y < … < university.degree |
| `pdays` | Binary `contacted_before` flag | 999 = never contacted; sparse otherwise |
| `month` / `day_of_week` | OHE | Seasonal and weekday patterns in subscription rates |
| `job`, `marital`, `contact`, `poutcome` | OHE | No natural ordinal order |
| `default`, `housing`, `loan` | Binary (yes=1) | Unknown treated as 0 (conservative) |
| Economic indicators | Keep as-is | Already numeric; high multicollinearity managed by RF |


In [ ]:
df = campaign.copy()

# ── Target encoding ────────────────────────────────────────────────────────────
df['y'] = (df['y'] == 'yes').astype(int)

# ── Ordinal education ─────────────────────────────────────────────────────────
EDUCATION_ORDER = [
    ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
     'high.school', 'professional.course', 'university.degree', 'unknown']
]
ord_enc = OrdinalEncoder(categories=EDUCATION_ORDER, handle_unknown='use_encoded_value', unknown_value=-1)
df['education_encoded'] = ord_enc.fit_transform(df[['education']])

# ── Binary flag: ever contacted before? ───────────────────────────────────────
df['contacted_before'] = (df['pdays'] != 999).astype(int)

# ── Binary yes/no columns (unknown → 0) ───────────────────────────────────────
for col in ['default', 'housing', 'loan']:
    df[col] = (df[col] == 'yes').astype(int)

# ── OHE categoricals ─────────────────────────────────────────────────────────
OHE_COLS = ['job', 'marital', 'contact', 'poutcome', 'month', 'day_of_week']
df = pd.get_dummies(df, columns=OHE_COLS, drop_first=False)

# ── Drop original columns no longer needed ─────────────────────────────────────
df = df.drop(columns=['education', 'pdays'])

print(f"Feature matrix shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


## 4. Train / Test Split

In [ ]:
FEATURE_COLS = [c for c in df.columns if c != 'y']
X = df[FEATURE_COLS]
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print(f"Train positive rate: {y_train.mean():.2%}")
print(f"Test  positive rate: {y_test.mean():.2%}")


## 5. Random Forest Classifier

**Why Random Forest?**
- Handles mixed feature types (numeric, binary, OHE) without scaling
- Built-in feature importance for interpretability
- Robust to class imbalance via `class_weight='balanced'`


In [ ]:
# ── Baseline RF ───────────────────────────────────────────────────────────────
rf_base = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)
print("Baseline Random Forest:")
print(classification_report(y_test, y_pred_base, target_names=['no', 'yes']))
print(f"AUC-ROC: {roc_auc_score(y_test, rf_base.predict_proba(X_test)[:, 1]):.4f}")


In [ ]:
# ── Hyperparameter tuning via GridSearchCV ────────────────────────────────────
param_grid = {
    'n_estimators':  [200, 400],
    'max_depth':     [10, 20, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features':  ['sqrt', 'log2'],
}

grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid, cv=5, scoring='f1', verbose=1, n_jobs=-1
)
grid_search.fit(X_train, y_train)

rf_best = grid_search.best_estimator_
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1:  {grid_search.best_score_:.4f}")


In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────────
y_pred  = rf_best.predict(X_test)
y_proba = rf_best.predict_proba(X_test)[:, 1]

print("Tuned Random Forest — Test Set:")
print(classification_report(y_test, y_pred, target_names=['no', 'yes']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['no', 'yes'], ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — Tuned Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Feature Importance

In [ ]:
fi = (
    pd.DataFrame({'feature': X_train.columns, 'importance': rf_best.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi['feature'][:20][::-1], fi['importance'][:20][::-1], color='steelblue')
ax.set_xlabel('Feature Importance', fontweight='bold')
ax.set_title('Top 20 Feature Importances — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(fi.head(10).to_string(index=False))


## 7. ROC Curve & Threshold Optimization

In a marketing campaign context, **recall on positives** (catching likely subscribers) may be more valuable than precision — contacting a non-subscriber wastes resources, but missing a subscriber loses revenue. The optimal threshold depends on the business cost ratio.


In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1], 'k--', lw=1.5, label='Random')
axes[0].set_xlabel('False Positive Rate', fontweight='bold')
axes[0].set_ylabel('True Positive Rate', fontweight='bold')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

# ── Precision-Recall curve ────────────────────────────────────────────────────
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx  = np.argmax(f1_scores)
best_thr  = pr_thresholds[best_idx] if best_idx < len(pr_thresholds) else 0.5

axes[1].plot(recall, precision, color='darkorange', lw=2)
axes[1].axvline(x=recall[best_idx], color='green', linestyle='--',
                label=f'Best F1 threshold={best_thr:.2f}')
axes[1].set_xlabel('Recall', fontweight='bold')
axes[1].set_ylabel('Precision', fontweight='bold')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Best F1 threshold: {best_thr:.2f}")
print(f"At this threshold → F1: {f1_scores[best_idx]:.4f}")


## 8. Limitations & Future Work

**Known limitations:**
- Economic indicators (`euribor3m`, `emp.var.rate`, `nr.employed`) are highly correlated — could cause instability; PCA or VIF analysis recommended
- `unknown` values in `default`, `housing`, `loan` are treated as `no` — could introduce bias
- No temporal split: the dataset covers multiple campaigns; a time-based split would be more realistic
- Random Forest is not easily deployable in real-time scoring pipelines vs. simpler models (Logistic Regression, LightGBM)

**Potential improvements:**
- Try LightGBM or XGBoost for better speed and comparable accuracy
- SMOTE or cost-sensitive learning to handle class imbalance more explicitly
- Build a cost-benefit analysis at different thresholds using estimated campaign cost vs. term deposit revenue
